# Mixed100K V2 — Colab 训练与评估

由实际运行的 `tiktok_techjam.ipynb` 整理，保留数据下载、训练、评估及 Drive 备份命令；已清空输出与运行元数据。完整方法与结果见仓库 README。

**数据和权重不在 GitHub 中。不要直接全部运行。** 若已在 `/content/ai_image_detector` 恢复项目与数据，请跳过相应下载/恢复步骤。先用 CPU 准备数据，训练/推理时再使用 GPU；更换运行时可能丢失 `/content`，请先备份。

## 0. 放置项目（任选一种方式）

- 从 GitHub 下载源码 ZIP，上传 Colab 并解压为 `/content/ai_image_detector`；私有仓库需先在浏览器登录下载，不要把访问令牌写进 notebook。
- 或使用下面的可选 Drive 恢复命令，将备份路径改成自己的文件。首次运行没有备份时，跳过这两格。

恢复命令只适合新运行时；已有项目时不要直接解压覆盖当前文件。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!cd /content && tar -xzf /content/drive/MyDrive/ai_image_detector_backup.tar.gz

## 1. 安装依赖

确认项目已位于 `/content/ai_image_detector`。Colab 不需要 `.venv`。

In [ ]:
!python -m pip install -r /content/ai_image_detector/requirements.txt
!python -m pip install -U modelscope-hub
!mkdir -p /content/ai_image_detector/data/wildfake_raw
!mkdir -p /content/ai_image_detector/data/wildfake_eval

## 2. 下载演示评估集（已有完整数据则跳过）

COCO val2017 4,998 张 Real + DALL·E Advanced/DALLE3 8,843 张 AI。**只做演示评估，不训练。** 下载完整 DALLE ZIP 可能较慢且需要较多磁盘。先取得这些图片，再准备训练集，可执行精确重叠检查。

In [ ]:
!MODELSCOPE_DOWNLOAD_PARALLEL_WORKERS=4 ms-hub download hy2628982280/WildFake "Images/Diffusion_based/DALLE.zip" "label_csv_files/dalle3.csv" --repo-type dataset --local-dir /content/ai_image_detector/data/wildfake_raw --max-workers 2

In [ ]:
!MODELSCOPE_DOWNLOAD_PARALLEL_WORKERS=4 ms-hub download hy2628982280/WildFake "Images/Real/coco.zip" "label_csv_files/real_coco.csv" --repo-type dataset --local-dir /content/ai_image_detector/data/wildfake_raw --max-workers 2

In [ ]:
!unzip -n -q /content/ai_image_detector/data/wildfake_raw/Images/Diffusion_based/DALLE.zip 'DALLE/Advanced/DALLE3/*' -d /content/ai_image_detector/data/wildfake_eval

In [ ]:
!unzip -n -q /content/ai_image_detector/data/wildfake_raw/Images/Real/coco.zip '*val2017/*' -d /content/ai_image_detector/data/wildfake_eval

检查图片数：以下两格预期分别输出 8843、4998。

In [ ]:
!find /content/ai_image_detector/data/wildfake_eval/DALLE/Advanced/DALLE3 \
  -type f \
  \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) \
  | wc -l

In [ ]:
!find /content/ai_image_detector/data/wildfake_eval \
  -type f \
  -path "*/val2017/*" \
  \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) \
  | wc -l

## 3. 准备 Mixed100K（已有完整数据则跳过）

训练 100,000 张 + 内部验证 4,000 张。CIFAKE/SID_Set 各 10,000 张训练图，其余来自 WildFake；排除 COCO/DALL·E 训练归档。

这里用 `config.mixed100k.yaml` 作为模板，避免旧 `config.yaml` 覆盖当前自动校准策略。重复运行可继续数据准备；不要在已有缓存中更改 seed/配额。下载不需要 GPU。

In [ ]:
%pip -q install kagglehub datasets modelscope-hub
!cd /content/ai_image_detector && python -u prepare_mixed_dataset.py --base-config config.mixed100k.yaml --workers 8

## 4. 训练（仅需测试已有权重时跳过）

4,000 张内部验证按固定 ID 分为 1,000 张阈值校准和 3,000 张模型评估。训练自动保存对应阈值。开始新实验前更换配置的 `output_dir`，或先备份已有权重，防止覆盖。

In [ ]:
!cd /content/ai_image_detector && python train.py --config config.mixed100k.yaml

## 5. 评估

先训练，或自行恢复可信的 `outputs/dinov2_mixed100k_v2/best.pt`。默认读取 checkpoint 阈值。当前保存模型的演示阈值为 `0.000005`；想临时指定可在命令末尾添加 `--threshold 0.000005`，不会修改 checkpoint。新模型不一定适用此阈值。

评估会生成独立的 `evaluation_runs/<run_id>/`，包括逐图预测、阈值对比及分数分布。此演示集曾用于人工阈值调整，结果不是独立最终测试成绩。

In [ ]:
# 新模型训练完成后，在 COCO / DALLE 测试集上评估
!cd /content/ai_image_detector && python evaluate.py --config config.mixed100k.yaml --checkpoint outputs/dinov2_mixed100k_v2/best.pt --source validation_demo

## 6. 可选：休息或更换运行时前备份

如果尚未挂载 Drive，先运行上面的 Drive 挂载格。下面备份包含代码和 outputs/权重，但 **排除 data/**；训练数据需要另行保存。请自行更改备份文件名，避免覆盖需要保留的旧备份。备份完成并确认文件存在后，再断开运行时以停止占用 GPU。

In [ ]:
!cd /content && tar \
  --exclude="ai_image_detector/data" \
  --exclude="ai_image_detector/__pycache__" \
  -czf /content/drive/MyDrive/ai_image_detector_backup.tar.gz \
  ai_image_detector

In [ ]:
!ls -lh /content/drive/MyDrive/ai_image_detector_backup.tar.gz